# LOAD LIBRARIES, SETUP REPRODUCIBILITIES, AND CONFIG

In [10]:
import os, json, math, random, csv
import time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from typing import List, Tuple
import matplotlib.pyplot as plt
import scipy.ndimage

import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Subset
import timm

# =====================================================
# Reproducibility & Config
# =====================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
TRAIN_IMG_DIR = Path("../Datasets/train/images/")
TRAIN_LBL_DIR = Path("../Datasets/train/labels/")
TEST_IMG_DIR  = Path("../Datasets/test/images/")
OUTPUT_DIR    = Path("../Models"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMIT_DIR    = Path("../Submission"); SUBMIT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_SUB    = Path("../Datasets/sample_submission.csv")

# DENSITY MAPS HELPERS

In [11]:
IM_SIZE   = 672
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

# --- Letterbox with metadata (scale & pad) ---
def letterbox_with_meta(im: Image.Image, max_side: int = IM_SIZE, fill: int = 0):
    w0, h0 = im.size
    scale = max_side / max(w0, h0)
    new_w, new_h = int(round(w0 * scale)), int(round(h0 * scale))
    im_resized = im.resize((new_w, new_h), resample=Image.BICUBIC, reducing_gap=1.0)
    pad_w, pad_h = max_side - new_w, max_side - new_h
    pad_left  = pad_w // 2
    pad_top   = pad_h // 2
    pad_right = pad_w - pad_left
    pad_bot   = pad_h - pad_top
    im_out = ImageOps.expand(im_resized, border=(pad_left, pad_top, pad_right, pad_bot), fill=fill)
    return im_out, scale, pad_left, pad_top, w0, h0

# --- Parse points from many JSON styles ---
def parse_points_any(d: dict) -> List[Tuple[float, float]]:
    pts = []
    for p in d.get("points", []):
        try:
            if isinstance(p, (list, tuple)) and len(p) >= 2:
                x = float(p[0]); y = float(p[1])
                pts.append((x, y))
            elif isinstance(p, dict) and "x" in p and "y" in p:
                x = float(p["x"]); y = float(p["y"])
                pts.append((x, y))
        except Exception:
            continue
    return pts

# --- Detect coord system & align to actual image size ---
def align_points_to_image(points: List[Tuple[float,float]], w0: int, h0: int) -> List[Tuple[float,float]]:
    """
    Heuristics:
      - If points look normalized (max <= ~1.5), treat as [0..1] and scale by (w0, h0).
      - Else if their max is far from (w0,h0), rescale per-axis by observed max.
      - Else assume already in pixel coords of original image.
    """
    if not points:
        return []

    xs = [p[0] for p in points]; ys = [p[1] for p in points]
    x_max, y_max = max(xs), max(ys)

    # Normalized case
    if x_max <= 1.5 and y_max <= 1.5:
        return [(x * w0, y * h0) for (x, y) in points]

    # Pixel but different base size (e.g., annotated on a different resolution)
    tol = 0.2  # 20% tolerance window
    if not ( (1 - tol) * w0 <= x_max <= (1 + tol) * w0 and (1 - tol) * h0 <= y_max <= (1 + tol) * h0 ):
        sx = w0 / max(x_max, 1e-6)
        sy = h0 / max(y_max, 1e-6)
        return [(x * sx, y * sy) for (x, y) in points]

    # Already in pixel coords of the current original image
    return points

# --- Apply letterbox transform to points ---
def apply_letterbox_to_points(points: List[Tuple[float,float]], scale: float, pad_left: int, pad_top: int):
    return [(x * scale + pad_left, y * scale + pad_top) for (x, y) in points]

# --- Build Gaussian density map ---
def make_density_map(H: int, W: int, points: List[Tuple[float,float]], sigma: float = 4.0):
    density = np.zeros((H, W), dtype=np.float32)
    for (x, y) in points:
        ix, iy = int(round(x)), int(round(y))
        if 0 <= ix < W and 0 <= iy < H:
            density[iy, ix] += 1.0
    # Gaussian blur; adjust sigma if scenes are very dense/sparse
    if sigma > 0:
        density = scipy.ndimage.gaussian_filter(density, sigma=sigma, mode="constant")
    return density


# DATASET

In [12]:
class CrowdCountDensityDataset(Dataset):
    def __init__(self, img_dir: Path, lbl_dir: Path, size=IM_SIZE, sigma=4.0, train=True):
        self.img_dir, self.lbl_dir, self.size, self.sigma, self.train = img_dir, lbl_dir, size, sigma, train
        self.img_files = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}])
        self.lbl_map = {p.stem: (lbl_dir / f"{p.stem}.json") for p in self.img_files}

    def __len__(self): return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        img = Image.open(img_path).convert("RGB")

        # 1) letterbox image and get transform
        img_lb, scale, pad_left, pad_top, w0, h0 = letterbox_with_meta(img, max_side=self.size)

        # 2) load JSON and parse points
        with open(self.lbl_map[img_path.stem], "r") as f:
            d = json.load(f)
        pts_raw = parse_points_any(d)

        # 3) align JSON points to original image size, then map through letterbox
        pts_px  = align_points_to_image(pts_raw, w0, h0)
        pts_t   = apply_letterbox_to_points(pts_px, scale, pad_left, pad_top)

        # 4) build density map
        density = make_density_map(self.size, self.size, pts_t, sigma=self.sigma)

        # (optional) simple aug
        if self.train and random.random() < 0.5:
            img_lb = ImageOps.mirror(img_lb)
            density = np.flip(density, axis=1).copy()

        # to tensors
        x = TF.to_tensor(img_lb)
        x = TF.normalize(x, NORM_MEAN, NORM_STD)
        y = torch.from_numpy(density).unsqueeze(0)  # [1,H,W]

        return x, y, img_path.name

# QUICK SANITY CHECK: IMAGE 1.JPG & 2.JPG

In [13]:
def debug_density_sum(stem: str, img_dir: Path, lbl_dir: Path, size=IM_SIZE, sigma=4.0):
    p_img  = img_dir / f"{stem}.jpg"
    p_json = lbl_dir / f"{stem}.json"

    img = Image.open(p_img).convert("RGB")
    img_lb, scale, pad_left, pad_top, w0, h0 = letterbox_with_meta(img, max_side=size)

    with open(p_json, "r") as f:
        d = json.load(f)

    # target count from JSON (human_num or len(points))
    gt_count = float(d["human_num"]) if "human_num" in d else float(len(d.get("points", [])))

    pts_raw = parse_points_any(d)
    pts_px  = align_points_to_image(pts_raw, w0, h0)
    pts_t   = apply_letterbox_to_points(pts_px, scale, pad_left, pad_top)
    dens    = make_density_map(size, size, pts_t, sigma=sigma)

    print(f"{stem}.jpg | JSON count = {gt_count:.2f} | density sum = {dens.sum():.2f} | #pts parsed = {len(pts_raw)}")

# your dirs:
TRAIN_IMG_DIR = Path("../Datasets/train/images/")
TRAIN_LBL_DIR = Path("../Datasets/train/labels/")

debug_density_sum("1", TRAIN_IMG_DIR, TRAIN_LBL_DIR)  # expect ~539
debug_density_sum("2", TRAIN_IMG_DIR, TRAIN_LBL_DIR)  # expect ~109


1.jpg | JSON count = 539.00 | density sum = 537.15 | #pts parsed = 539
2.jpg | JSON count = 109.00 | density sum = 108.55 | #pts parsed = 109


# TRAIN AND VALIDATION DATASET SPLIT

In [14]:
full_ds = CrowdCountDensityDataset(TRAIN_IMG_DIR, TRAIN_LBL_DIR, train=True)
counts = []
for p in full_ds.img_files:
    with open(full_ds.lbl_map[p.stem], "r") as f:
        d = json.load(f)
    counts.append(float(d["human_num"]) if "human_num" in d else len(d["points"]))
counts = np.array(counts)

bins = np.digitize(counts, np.quantile(counts, [0.2,0.4,0.6,0.8]))
indices = np.arange(len(full_ds))
val_ratio = 0.1
val_idx = []
for b in np.unique(bins):
    b_idx = indices[bins==b]
    k = max(1, int(round(len(b_idx)*val_ratio)))
    val_idx.extend(np.random.choice(b_idx, size=k, replace=False))
val_idx = np.array(sorted(set(val_idx)))
train_idx = np.array(sorted(list(set(indices)-set(val_idx))))

ds_tr = Subset(full_ds, train_idx.tolist())
ds_va = Subset(full_ds, val_idx.tolist())

BATCH_SIZE = 2
NUM_WORKERS = min(2, os.cpu_count() or 2)

dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train datasets = {len(ds_tr)}, Validation datasets = {len(ds_va)}")

Train datasets = 1710, Validation datasets = 190


# MODELLING: SWIN-T REGRESSOR + SMALL DECODER

In [21]:
class SwinDensityNet(nn.Module):
    def __init__(self, img_size=IM_SIZE, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            features_only=True,
            img_size=img_size,
            strict_img_size=False,
            out_indices=(3,)  # deepest stage
        )
        in_ch = self.backbone.feature_info[-1]['num_chs']
        self.decoder = nn.Sequential(
            nn.Conv2d(in_ch, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 1, 1)
        )

    def forward(self, x):
        feats = self.backbone(x)[-1]   # expected [B, C, H, W]
        if feats.ndim == 4 and feats.shape[1] != self.backbone.feature_info[-1]['num_chs']:
            # If it came out as [B, H, W, C], permute back
            feats = feats.permute(0, 3, 1, 2).contiguous()
        out = self.decoder(feats)
        out = nn.functional.interpolate(out, size=x.shape[2:], mode="bilinear", align_corners=False)
        return out


# LOSS & METRICS SETUP

In [22]:
loss_fn = nn.MSELoss()

@torch.no_grad()
def batch_mae_density(pred_map, gt_map):
    pred_cnt = pred_map.sum(dim=[1,2,3])
    gt_cnt   = gt_map.sum(dim=[1,2,3])
    return (pred_cnt - gt_cnt).abs().sum().item()

# OPTIMIZER AND SCHEDULER

In [23]:
LR, WD = 1e-4, 5e-2
EPOCHS, WARMUP_EPOCHS = 20, 3
GRAD_CLIP = 1.0

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
def lr_at(epoch):
    if epoch < WARMUP_EPOCHS: return (epoch+1)/WARMUP_EPOCHS
    t = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5*(1+math.cos(math.pi*t))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_at)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))

/tmp/ipykernel_158884/2984329432.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))


# TRAINING LOOP

In [24]:
BEST_PATH = OUTPUT_DIR / "best_swin_density_new.pth"
history = {'tr_loss':[], 'tr_mae':[], 'va_loss':[], 'va_mae':[], 'lr':[]}
best_mae, patience, bad_epochs = float("inf"), 8, 0

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    tr_loss=tr_mae=0.0; n_tr=0
    for x,y,_ in dl_tr:
        x,y=x.to(device), y.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            pred=model(x); loss=loss_fn(pred,y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        tr_loss+=loss.item()*x.size(0); tr_mae+=batch_mae_density(pred,y); n_tr+=x.size(0)
    tr_loss/=n_tr; tr_mae/=n_tr

    model.eval(); va_loss=va_mae=0.0; n_va=0
    with torch.inference_mode(), torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
        for x,y,_ in dl_va:
            x,y=x.to(device), y.to(device)
            pred=model(x); loss=loss_fn(pred,y)
            va_loss+=loss.item()*x.size(0); va_mae+=batch_mae_density(pred,y); n_va+=x.size(0)
    va_loss/=n_va; va_mae/=n_va

    sched.step()
    history['tr_loss'].append(tr_loss); history['tr_mae'].append(tr_mae)
    history['va_loss'].append(va_loss); history['va_mae'].append(va_mae)
    history['lr'].append(sched.get_last_lr()[0])

    print(f"Epoch: {epoch+1:02d}/{EPOCHS} | Train Loss: {tr_loss:.4f} MAE: {tr_mae:.2f} | "
          f"Val Loss: {va_loss:.4f} MAE: {va_mae:.2f} | LR: {sched.get_last_lr()[0]:.2e} "
          f"Time per epochs {time.time()-t0:.2f} sec")

    if va_mae < best_mae - 1e-6:
        best_mae, bad_epochs = va_mae, 0
        torch.save({"model": model.state_dict()}, BEST_PATH)
    else:
        bad_epochs+=1
        if bad_epochs>=patience: print("Early stopping."); break

print("Best val MAE:", best_mae)

/tmp/ipykernel_158884/2111314837.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):


RuntimeError: Given groups=1, weight of size [256, 768, 3, 3], expected input[2, 21, 21, 768] to have 768 channels, but got 21 channels instead